# An Argus night, live

The field movie, drawn live: a real field observed for one night by simulated Argus and processed by hdps (reduction, WCS, reprojection onto its sky tiles, SFFT difference imaging against a deep reference, detection, alert features), with **variable stars that you place and edit**: none to 100 of them, labelled A, B, …, Z, AA, … in their own colours.

* **Field**: f294 · Aquila (the videos' field); f065 · Perseus, Milky Way plane; f098 · Lynx, away from the plane, each one precomputed hdps night.
* **Night**: shown for a field observed on several nights (a week of M67); a change of night keeps your stars.
* **Band**: β or ρ.
* **Your stars**: the list holds them; pick one to edit it. **Add** places a star of the type shown at a quiet free place, **Duplicate** copies the selected star, **Remove** takes it out, **Clear all** removes every star, **One of each type** adds one star of every type at quiet places well apart. The app opens with one star, A.
* **Star type** and its sliders edit the selected star: magnitude, amplitude, period (a transient: its timescale).
* **Click** the exposure or the difference image to move the selected star there.
* **Airmass +**: observe the same night lower in the sky.
* **Zoom**: magnify the exposure and difference images about the selected star (1–8×).
* **Show alerts**: the alert marks on or off; **Loop**: play the night again from its start.
* **Play** runs through the night at the chosen speed; the frame slider goes to any exposure.

The light-curve, CMD and SED panels show up to six stars: the selected one, then your most recently edited, then the field's own variables. Every star of yours seen in the images has its circle, label and alert marks. The status line gives the selected star, then the number of your stars and the alerts per exposure from them and from the field.

Everything except your stars keeps its photometry and alert candidates from the real hdps run. Your stars are measured on hdps's images with hdps's own detector and alert detection, all of them in one pass per exposure; the note under the controls lists what is simplified for them.

**Real/bogus is perfect** (truth, no model): an alert is raised only where a star we simulated, so known to be real, is detected. Each of your stars alerts in every exposure where hdps's detection finds it; of the field's candidates, only those on its simulated or known variables alert.

### Running it

Run the two code cells below in order.

1. **Setup** finds the demo's archive (its code and data, 308 MB) in the symposium bucket, checks it, and unpacks
   it once into `~/.cache/argus_night_live/`; later runs skip straight to checking the Python packages. If a package
   is missing it installs it with `pip install --user` and asks you to restart the kernel: do that
   (Kernel → Restart Kernel), reload the browser page, and run both cells again.
2. **The app** starts in about 40 s the first time (it writes an uncompressed copy of the field, about the size of
   the archive's share for it, next to the unpacked files), faster after that. The first change of a control is
   slower than the rest (hdps's detector is compiled); a change re-measures your stars in every exposure of the night.

In [1]:
# Setup: find the demo's archive, check it, unpack it once, check the Python packages. Safe to run again.
ARCHIVE = 'argus_night_live_20261027T2010_6199055.tar'
SHA256 = 'c04a93ff16a8fdb956a9e4651ec49b9ef7a97c00b2fc3dd47d6894da7ba594eb'
VERSION = '20261027T2010_6199055'
BUCKET = 'schmidt-observatory-system'
S3_KEY = 'argus/argus_night_live/argus_night_live_20261027T2010_6199055.tar'


import hashlib, importlib, importlib.util, os, shutil, site, subprocess, sys, tarfile, tempfile, time
from pathlib import Path


class SetupStop(Exception):
    """Stops the notebook here (and Run All with it) with a plain message instead of a traceback."""

    def _render_traceback_(self):
        return ["", "*** Setup stopped ***", *str(self).splitlines(), ""]


HOME = Path(os.environ.get("ARGUS_NIGHT_LIVE_HOME") or "~/.cache/argus_night_live").expanduser()
APP_DIR = HOME / VERSION
MARKER = APP_DIR / ".unpacked"  # written last: holds SHA256 once everything is in place


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()


def _checked(path, where, tried):
    """path if it is the archive (sha256 matches), else None, noting why in tried."""
    path = Path(path).expanduser()
    if not path.is_file():
        tried.append(f"{where}: no {path}")
        return None
    print(f"found {path} ({where}, {path.stat().st_size / 1e6:.0f} MB); checking its sha256 ...", flush=True)
    if _sha256(path) != SHA256:
        tried.append(f"{where}: {path} is not this archive (sha256 differs: another version, or incomplete)")
        print("  sha256 differs: not using it", flush=True)
        return None
    return path


def _from_s3(tried):
    """The archive downloaded through the S3 API into HOME, or None."""
    url = f"s3://{BUCKET}/{S3_KEY}"
    try:
        import fsspec

        fs = fsspec.filesystem("s3")
        if not fs.exists(url):
            tried.append(f"S3 API: no {url}")
            return None
        HOME.mkdir(parents=True, exist_ok=True)
        part = HOME / (ARCHIVE + ".part")
        print(f"downloading {url} to {HOME} ...", flush=True)
        t = time.time()
        fs.get(url, str(part))
        print(f"  {part.stat().st_size / 1e6:.0f} MB in {time.time() - t:.0f} s", flush=True)
        part.replace(HOME / ARCHIVE)
    except Exception as e:  # no s3fs, no credentials, no network
        tried.append(f"S3 API: {url}: {type(e).__name__}: {e}")
        return None
    return _checked(HOME / ARCHIVE, "downloaded", tried)


def find_archive():
    """(path, downloaded): the archive, from the first place that has it."""
    tried = []
    if os.environ.get("ARGUS_NIGHT_LIVE_ARCHIVE"):
        if p := _checked(os.environ["ARGUS_NIGHT_LIVE_ARCHIVE"], "ARGUS_NIGHT_LIVE_ARCHIVE", tried):
            return p, False
    if p := _checked(f"~/s3/{BUCKET}/{S3_KEY}", "the hub's bucket mount", tried):
        return p, False
    if p := _checked(HOME / ARCHIVE, "downloaded before", tried):
        return p, True
    if p := _from_s3(tried):
        return p, True
    if p := _checked(Path.cwd() / ARCHIVE, "next to the notebook", tried):
        return p, False
    raise SetupStop("\n".join([f"Could not find {ARCHIVE} (sha256 {SHA256[:12]}...). Looked:", *("  " + t for t in tried),
                               "Put the archive next to this notebook, or set ARGUS_NIGHT_LIVE_ARCHIVE to its path, and run this cell again."]))


def unpack(archive):
    HOME.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(HOME).free
    need = 2.5 * archive.stat().st_size + 2e8  # the files, and the field caches the app writes (about the packs' size)
    if free < need:
        print(f"WARNING: {free / 1e9:.1f} GB free in {HOME}; the app wants about {need / 1e9:.1f} GB "
              f"(set ARGUS_NIGHT_LIVE_HOME to a folder with more room)", flush=True)
    print(f"unpacking into {APP_DIR} ...", flush=True)
    t = time.time()
    tmp = Path(tempfile.mkdtemp(prefix=f".{VERSION}.", dir=HOME))  # unpacked aside, moved into place when whole
    try:
        with tarfile.open(archive) as tar:
            tar.extractall(tmp, filter="data")
        if not (tmp / VERSION / "argus_night_live" / "__init__.py").is_file():
            raise SetupStop(f"{archive} does not hold {VERSION}/argus_night_live/: not this demo's archive")
        if APP_DIR.exists():  # an earlier unpack that did not finish
            shutil.rmtree(APP_DIR)
        (tmp / VERSION).rename(APP_DIR)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    MARKER.write_text(SHA256 + "\n")
    print(f"  done in {time.time() - t:.0f} s", flush=True)




print(f"argus_night_live {VERSION}; Python {sys.version.split()[0]} ({sys.executable})", flush=True)
if sys.version_info < (3, 12):
    raise SetupStop(f"needs Python 3.12 or newer (tested with 3.13 and 3.14); this kernel is {sys.version.split()[0]}")
if MARKER.is_file() and MARKER.read_text().strip() == SHA256:
    print(f"already unpacked in {APP_DIR}", flush=True)
else:
    archive, downloaded = find_archive()
    unpack(archive)
    if downloaded:  # not needed again: the marker says the unpacked copy is whole
        archive.unlink(missing_ok=True)

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
importlib.invalidate_caches()
loaded = sys.modules.get("argus_night_live")
if loaded is not None and not str(getattr(loaded, "__file__", "")).startswith(str(APP_DIR)):
    raise SetupStop(f"argus_night_live is already imported from {loaded.__file__}; restart the kernel and run the cells again")
print("Ready: run the next cell.", flush=True)


argus_night_live 20261027T2010_6199055; Python 3.13.13 (/srv/conda/envs/notebook/bin/python)
already unpacked in /home/jovyan/.cache/argus_night_live/20261027T2010_6199055
Ready: run the next cell.


In [2]:
%matplotlib widget
import warnings
warnings.simplefilter("ignore", RuntimeWarning)
from argus_night_live.app import App

app = App()
app.show()

2026-10-27 21:23:17,355 — argus_sim — INFO — <module>:51 — No configuration file found, proceeding with defaults.
